# IFEST DAC 2026 — V5 Asymmetric High-Confidence Global Rescue

**Agreed V5-A design only. No new Transformer training.**

V5 freezes the public-LB **V4 champion (0.95681)** and adds only the previously-audited asymmetric rescue package:

1. Reconstruct the frozen V2 train→test graph exactly.
2. Reconstruct the frozen V4 test↔test canonical matching exactly.
3. Preserve every V2 graph-forced and V4 canonical decision exactly.
4. Consider only rows that are still pure `v2_fallback`, currently predicted `0` by V4.
5. Rescue `0 → 1` only when **V3 blend probability ≥ 0.67** and **V2 global entailment ≥ 0.50**.
6. Never perform `1 → 0` semantic overrides.

The notebook embeds the frozen V2/V3 score artifacts, so it runs directly from the competition CSVs in Kaggle. It intentionally performs **no model training, no model download, no fuzzy graph expansion, no threshold search for V5, and no leaderboard-tuned ID hardcoding**.

Validation is fixed to the agreed methodology: component-disjoint transductive folds plus 100 repeated pseudo-competition splits. V5 must pass hard invariants before a submission is exported.

In [1]:
import os, re, io, json, gzip, base64, hashlib, unicodedata, warnings, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)

# ---------------- V5 FROZEN CONFIG ----------------
SEED = 2026
N_FOLDS = 3
N_PSEUDO_SPLITS = 100

# Frozen V2 graph package.
V2_TITLE_GATE = 0.9839511586057019
MIN_GRAPH_CONTENT_WORDS = 20

# Frozen V4 transductive package.
RANK_V3_WEIGHT = 0.45
RANK_V2_GLOBAL_WEIGHT = 0.55
CONTENT_CANONICAL_MIN_SCORE = 0.20
TITLE_CANONICAL_MIN_SCORE = 0.90
TITLE_CANONICAL_MIN_MARGIN = 0.02

# Frozen V5 rescue package — do not retune in this notebook.
V5_BLEND_MIN = 0.67
V5_GLOBAL_ENTAIL_MIN = 0.50

PUBLIC_LB_V1 = 0.93367
PUBLIC_LB_V2 = 0.94856
PUBLIC_LB_V3 = 0.94856
PUBLIC_LB_V4 = 0.95681

EXPECTED_TRAIN_ROWS = 14397
EXPECTED_TEST_ROWS = 3603
EXPECTED_V4_CHANGES_VS_V2 = 16
EXPECTED_V5_RESCUES = 4
EXPECTED_V5_DIRECTION = "0_to_1_only"

EXPECTED_DATA_HASHES = {
    "train": "d2c3cf75cb6ef8d502f32d9b8ed8b22508040ea8570a295269a53b67e1e184ab",
    "test": "0871476c2c74ee5ba6daa931dbd6cfe1d3a992ac3da9c4ea5228c1deed16cf88",
    "sample_submission": "2725f8fe1c074b5e1d7c7f1234cfa482871d2ea94f6056b4451c22552809ebf5",
}

OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "/mnt/data/ifest_v5_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("IFEST DAC V5 — Asymmetric High-Confidence Global Rescue")
print("Training: intentionally disabled by design")
print(f"Rescue gate: V4=0 & v2_fallback & V3_blend>={V5_BLEND_MIN:.2f} & V2_global_E>={V5_GLOBAL_ENTAIL_MIN:.2f}")
try:
    import torch
    print(f"PyTorch={torch.__version__} | CUDA={torch.cuda.is_available()} | GPUs={torch.cuda.device_count()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(i, torch.cuda.get_device_name(i))
except Exception as e:
    print("Torch diagnostic unavailable:", repr(e))

IFEST DAC V5 — Asymmetric High-Confidence Global Rescue
Training: intentionally disabled by design
Rescue gate: V4=0 & v2_fallback & V3_blend>=0.67 & V2_global_E>=0.50
PyTorch=2.10.0+cu128 | CUDA=True | GPUs=2
0 Tesla T4
1 Tesla T4


In [2]:
FROZEN_ARTIFACTS = {
    'V2_OOF': {
        'sha256': '43658c849e5837002646dfa042083f420b808a05f0989f49788ffbcdcd8110ab',
        'rows': 14397,
        'columns': ['id', 'label', 'meta_prob', 'global_e', 'local_max_c'],
        'payload': (
            '...'
        ),
    },
    'V2_TEST': {
        'sha256': '99b09b02c7b611cd009a1a08a30588157412c423ff5d8853691ee883b2981a81',
        'rows': 3603,
        'columns': ['id', 'meta_prob', 'global_e', 'local_max_c', 'graph_rule', 'forced_label', 'label'],
        'payload': (
            '...'
        ),
    },
    'V3_OOF': {
        'sha256': 'c63b45e870f66ba64b6b6e00a88a4121660b2a02d34d9298ca834a297532d760',
        'rows': 14397,
        'columns': ['id', 'label', 'v1_meta_prob', 'v2_meta_prob', 'blend_prob'],
        'payload': (
            '...'
        ),
    },
    'V3_TEST': {
        'sha256': 'c98cc4d8a28272128b91ef9f9b7bdbddaff31e09bad4b6bfc31d95273929a146',
        'rows': 3603,
        'columns': ['id', 'v1_meta_prob', 'v2_meta_prob', 'blend_prob'],
        'payload': (
            '...'
        ),
    },
}


def load_frozen_csv(name):
    spec = FROZEN_ARTIFACTS[name]
    raw = gzip.decompress(base64.b64decode(spec["payload"]))
    digest = hashlib.sha256(raw).hexdigest()
    if digest != spec["sha256"]:
        raise RuntimeError(f"Frozen artifact checksum mismatch for {name}: {digest}")
    df = pd.read_csv(io.BytesIO(raw))
    if len(df) != spec["rows"] or df.columns.tolist() != spec["columns"]:
        raise RuntimeError(f"Frozen artifact schema mismatch for {name}")
    print(f"Loaded {name}: {df.shape} sha256={digest[:12]}...")
    return df

v2_oof = load_frozen_csv("V2_OOF")
v2_test = load_frozen_csv("V2_TEST")
v3_oof = load_frozen_csv("V3_OOF")
v3_test = load_frozen_csv("V3_TEST")


Loaded V2_OOF: (14397, 5) sha256=43658c849e58...
Loaded V2_TEST: (3603, 7) sha256=99b09b02c7b6...
Loaded V3_OOF: (14397, 5) sha256=c63b45e870f6...
Loaded V3_TEST: (3603, 4) sha256=c98cc4d8a282...


In [3]:
def find_competition_csv(filename):
    roots = [Path("/kaggle/input"), Path("/mnt/data/ifest_dac")]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        candidates.extend(root.rglob(filename))
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename} under Kaggle input")
    # Prefer the competition dataset path when present.
    candidates = sorted(candidates, key=lambda p: ("penyisihan-ifest-2026-dac" not in str(p).lower(), len(str(p))))
    return candidates[0]


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

train_path = find_competition_csv("train.csv")
test_path = find_competition_csv("test.csv")
sample_path = find_competition_csv("sample_submission.csv")

dataset_hash_mismatches = {}
for name, path in [("train", train_path), ("test", test_path), ("sample_submission", sample_path)]:
    digest = sha256_file(path)
    print(name, path, digest)
    if digest != EXPECTED_DATA_HASHES[name]:
        dataset_hash_mismatches[name] = {"expected": EXPECTED_DATA_HASHES[name], "actual": digest}

# ponytail: Kaggle mirrors may repackage identical competition data.
# Artifact alignment, V2 graph parity, and the final V4-change invariant below remain hard guards.
if dataset_hash_mismatches:
    print(f"WARNING: Dataset file hashes differ from the frozen snapshot: {dataset_hash_mismatches}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
assert train.shape[0] == EXPECTED_TRAIN_ROWS and test.shape[0] == EXPECTED_TEST_ROWS
assert train.columns.tolist() == ["id", "title", "content", "label"]
assert test.columns.tolist() == ["id", "title", "content"]
assert sample.columns.tolist() == ["id", "label"]
assert sample.id.astype(str).tolist() == test.id.astype(str).tolist()
assert set(train.label.unique()) == {0, 1}

# Frozen artifact alignment is mandatory.
assert v2_oof.id.astype(str).tolist() == train.id.astype(str).tolist()
assert v3_oof.id.astype(str).tolist() == train.id.astype(str).tolist()
assert v2_test.id.astype(str).tolist() == test.id.astype(str).tolist()
assert v3_test.id.astype(str).tolist() == test.id.astype(str).tolist()
assert np.array_equal(v2_oof.label.to_numpy(dtype=int), train.label.to_numpy(dtype=int))
assert np.array_equal(v3_oof.label.to_numpy(dtype=int), train.label.to_numpy(dtype=int))

print("Dataset and frozen-artifact alignment: OK")
print("train", train.shape, "test", test.shape)
print("train label counts", train.label.value_counts().sort_index().to_dict())

train /kaggle/input/competitions/penyisihan-ifest-2026-dac/train.csv a4ba91606302455da8a9523906a25361d581ac15b108964cdb3ab6beb0adb5d8
test /kaggle/input/competitions/penyisihan-ifest-2026-dac/test.csv 8e5de5bf53a50df87270dccdaba5ed0700d3729b5b03e4ce7724ac1769177005
sample_submission /kaggle/input/competitions/penyisihan-ifest-2026-dac/sample_submission.csv ea09389db317450c7b93da76d87b771daca79e66716b84b7432f1653a67582c6
Dataset and frozen-artifact alignment: OK
train (14397, 4) test (3603, 3)
train label counts {0: 1437, 1: 12960}


## Exact normalization and frozen V2 graph parity

In [4]:
TOKEN_RE = re.compile(r"[\w]+", flags=re.UNICODE)
SOURCE_SUFFIX_RE = re.compile(r"\s+kumparan\s+com\s*$", re.I)


def clean_ws(text):
    text = unicodedata.normalize("NFKC", str(text)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()


def clean_title_for_model(text):
    return SOURCE_SUFFIX_RE.sub("", clean_ws(text)).strip()


def normalize_key(text):
    return " ".join(TOKEN_RE.findall(clean_ws(text).lower()))

for df in (train, test):
    df["title_model"] = df.title.map(clean_title_for_model)
    df["title_key"] = df.title_model.map(normalize_key)
    df["content_key"] = df.content.map(normalize_key)
    df["pair_key"] = df.title_key + "|||" + df.content_key
    df["content_words"] = df.content.map(lambda x: len(clean_ws(x).split()))


def build_graph_reference(ref_df):
    pair_to_label = {}
    for key, g in ref_df.groupby("pair_key", sort=False):
        labs = set(map(int, g.label))
        if len(labs) == 1:
            pair_to_label[key] = next(iter(labs))
    content_to_pos_titles = defaultdict(set)
    title_to_pos_contents = defaultdict(set)
    for r in ref_df.loc[ref_df.label == 1].itertuples():
        content_to_pos_titles[r.content_key].add(r.title_key)
        title_to_pos_contents[r.title_key].add(r.content_key)
    return pair_to_label, content_to_pos_titles, title_to_pos_contents


def frozen_v2_graph_decisions(query_df, ref_df, v2_meta_prob):
    query_df = query_df.reset_index(drop=True)
    pair_to_label, content_to_pos_titles, title_to_pos_contents = build_graph_reference(ref_df)
    rules = np.array(["semantic"] * len(query_df), dtype=object)
    forced = np.full(len(query_df), -1, dtype=np.int8)
    v2_meta_prob = np.asarray(v2_meta_prob, dtype=float)
    if len(v2_meta_prob) != len(query_df):
        raise ValueError("v2_meta_prob/query length mismatch")

    for pos, r in enumerate(query_df.itertuples()):
        if r.pair_key in pair_to_label:
            forced[pos] = int(pair_to_label[r.pair_key])
            rules[pos] = "exact_pair"
            continue

        malformed = int(r.content_words) < MIN_GRAPH_CONTENT_WORDS
        pos_titles = content_to_pos_titles.get(r.content_key, set())
        if not malformed and len(pos_titles) == 1 and r.title_key not in pos_titles:
            forced[pos] = 0
            rules[pos] = "unique_content_different_title"
            continue

        pos_contents = title_to_pos_contents.get(r.title_key, set())
        if (
            not malformed and pos_contents and r.content_key not in pos_contents
            and v2_meta_prob[pos] < V2_TITLE_GATE
        ):
            forced[pos] = 0
            rules[pos] = "gated_title_different_content"

    return rules, forced

v2_test_meta = v2_test.meta_prob.to_numpy(dtype=float)
reconstructed_v2_rule, reconstructed_v2_forced = frozen_v2_graph_decisions(test, train, v2_test_meta)
assert np.array_equal(reconstructed_v2_rule, v2_test.graph_rule.astype(str).to_numpy()), "Frozen V2 graph rule parity failure"
assert np.array_equal(reconstructed_v2_forced, v2_test.forced_label.to_numpy(dtype=np.int8)), "Frozen V2 forced-label parity failure"
print("Frozen V2 graph parity: EXACT MATCH")
display(pd.Series(reconstructed_v2_rule).value_counts().to_frame("rows"))

Frozen V2 graph parity: EXACT MATCH


,rows
semantic,2447
exact_pair,927
unique_content_different_title,212
gated_title_different_content,17


## Structural profile — frozen V4 prior check

In [5]:
def structural_profile(df):
    rows = []
    # Same content, multiple distinct titles.
    for content_key, g in df.groupby("content_key", sort=False):
        n_titles = g.title_key.nunique()
        if n_titles < 2:
            continue
        pos_titles = g.loc[g.label == 1, "title_key"].nunique()
        rows.append({"axis": "content", "group_key": content_key, "candidates": int(n_titles), "positive_candidates": int(pos_titles), "rows": len(g)})
    # Same title, multiple distinct contents.
    for title_key, g in df.groupby("title_key", sort=False):
        n_contents = g.content_key.nunique()
        if n_contents < 2:
            continue
        pos_contents = g.loc[g.label == 1, "content_key"].nunique()
        rows.append({"axis": "title", "group_key": title_key, "candidates": int(n_contents), "positive_candidates": int(pos_contents), "rows": len(g)})
    return pd.DataFrame(rows)

profile = structural_profile(train)
profile_summary = (
    profile.groupby("axis")
    .agg(groups=("group_key", "size"), exactly_one_positive=("positive_candidates", lambda s: int((s == 1).sum())), zero_positive=("positive_candidates", lambda s: int((s == 0).sum())), multi_positive=("positive_candidates", lambda s: int((s > 1).sum())))
    .reset_index()
)
profile_summary["exactly_one_positive_rate"] = profile_summary.exactly_one_positive / profile_summary.groups
print("Train structural profile")
display(profile_summary)
content_rate = float(profile_summary.loc[profile_summary.axis == "content", "exactly_one_positive_rate"].iloc[0])
assert content_rate >= 0.97, f"Same-content canonical prior unexpectedly weakened: {content_rate:.4f}"

Train structural profile


,axis,groups,exactly_one_positive,zero_positive,multi_positive,exactly_one_positive_rate
0,content,893,873,16,4,0.977604
1,title,121,117,0,4,0.966942


## Frozen score construction, V4 package, and V5 rescue

In [6]:
# V2 remains the final-classification anchor. V3/global scores are used only for relative candidate ranking.
v2_oof_meta = v2_oof.meta_prob.to_numpy(dtype=float)
v2_test_meta = v2_test.meta_prob.to_numpy(dtype=float)
v2_oof_global = v2_oof.global_e.to_numpy(dtype=float)
v2_test_global = v2_test.global_e.to_numpy(dtype=float)
v3_oof_blend = v3_oof.blend_prob.to_numpy(dtype=float)
v3_test_blend = v3_test.blend_prob.to_numpy(dtype=float)

rank_oof = RANK_V3_WEIGHT * v3_oof_blend + RANK_V2_GLOBAL_WEIGHT * v2_oof_global
rank_test = RANK_V3_WEIGHT * v3_test_blend + RANK_V2_GLOBAL_WEIGHT * v2_test_global
assert np.isfinite(rank_oof).all() and np.isfinite(rank_test).all()
assert ((rank_oof >= 0) & (rank_oof <= 1)).all()
assert ((rank_test >= 0) & (rank_test <= 1)).all()

# Recover the frozen V2 semantic threshold only for validation reconstruction.
# Actual test labels are read directly from the already-submitted V2 artifact.
def exact_macro_f1_threshold(y, probs):
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(probs, dtype=float)
    order = np.argsort(p, kind="mergesort")
    ps = p[order]; ys = y[order]
    unique_vals, first_idx = np.unique(ps, return_index=True)
    candidates = [np.nextafter(float(ps.min()), -np.inf)]
    candidates.extend(float(x) for x in unique_vals)
    candidates.append(np.nextafter(float(ps.max()), np.inf))
    best_t, best_score = 0.5, -1.0
    for t in candidates:
        pred = (p >= t).astype(np.int8)
        score = f1_score(y, pred, average="macro", zero_division=0)
        if score > best_score + 1e-15 or (abs(score - best_score) <= 1e-15 and abs(t - 0.5) < abs(best_t - 0.5)):
            best_t, best_score = float(t), float(score)
    return best_t, best_score

V2_META_THRESHOLD, v2_oof_macro = exact_macro_f1_threshold(train.label.to_numpy(dtype=np.int8), v2_oof_meta)
print(f"Frozen V2 meta threshold recovered from OOF: {V2_META_THRESHOLD:.9f} | macro_f1={v2_oof_macro:.6f}")


def apply_transductive_v4(query_df, base_labels, base_forced, rank_scores, ref_df):
    """Apply only the pre-agreed V4 package. V2-forced rows are immutable."""
    q = query_df.reset_index(drop=True).copy()
    base_labels = np.asarray(base_labels, dtype=np.int8)
    base_forced = np.asarray(base_forced, dtype=np.int8)
    rank_scores = np.asarray(rank_scores, dtype=float)
    n = len(q)
    if not (len(base_labels) == len(base_forced) == len(rank_scores) == n):
        raise ValueError("V4 input length mismatch")

    _, content_to_pos_titles, title_to_pos_contents = build_graph_reference(ref_df)
    labels = base_labels.copy()
    v4_forced = np.full(n, -1, dtype=np.int8)
    v4_rule = np.array(["v2_fallback"] * n, dtype=object)
    locked = base_forced >= 0
    v4_rule[locked] = "v2_graph_locked"
    group_records = []

    # -------- 1) SAME CONTENT / DIFFERENT TITLES --------
    available = np.where(~locked)[0]
    if len(available):
        aq = q.iloc[available].copy()
        aq["_pos"] = available
        for content_key, g in aq.groupby("content_key", sort=False):
            if int(g.content_words.iloc[0]) < MIN_GRAPH_CONTENT_WORDS:
                continue
            if g.title_key.nunique() < 2:
                continue
            # V4 acts only when train does not already identify any positive title for this body.
            if len(content_to_pos_titles.get(content_key, set())) > 0:
                continue

            cand = (
                g.assign(_score=rank_scores[g._pos.to_numpy(dtype=int)])
                .groupby("title_key", as_index=False)
                .agg(score=("_score", "mean"), rows=("_pos", "size"))
                .sort_values(["score", "title_key"], ascending=[False, True], kind="mergesort")
                .reset_index(drop=True)
            )
            top_score = float(cand.score.iloc[0])
            if top_score < CONTENT_CANONICAL_MIN_SCORE:
                group_records.append({"axis":"content", "group_key":content_key, "candidates":len(cand), "top_score":top_score, "margin":np.nan, "applied":False, "top_key":cand.title_key.iloc[0]})
                continue
            top_key = cand.title_key.iloc[0]
            pos_idx = g._pos.to_numpy(dtype=int)
            new = (g.title_key.to_numpy() == top_key).astype(np.int8)
            labels[pos_idx] = new
            v4_forced[pos_idx] = new
            v4_rule[pos_idx] = "test_content_canonical"
            locked[pos_idx] = True
            group_records.append({"axis":"content", "group_key":content_key, "candidates":len(cand), "top_score":top_score, "margin":np.nan, "applied":True, "top_key":top_key})

    # -------- 2) SAME TITLE / DIFFERENT CONTENTS --------
    available = np.where(~locked)[0]
    if len(available):
        aq = q.iloc[available].copy()
        aq["_pos"] = available
        for title_key, g in aq.groupby("title_key", sort=False):
            if g.content_key.nunique() < 2:
                continue
            # If train knows a positive content for this title, V2 train->test graph owns the decision.
            if len(title_to_pos_contents.get(title_key, set())) > 0:
                continue

            cand = (
                g.assign(_score=rank_scores[g._pos.to_numpy(dtype=int)])
                .groupby("content_key", as_index=False)
                .agg(score=("_score", "mean"), rows=("_pos", "size"))
                .sort_values(["score", "content_key"], ascending=[False, True], kind="mergesort")
                .reset_index(drop=True)
            )
            top_score = float(cand.score.iloc[0])
            second_score = float(cand.score.iloc[1])
            margin = top_score - second_score
            if top_score < TITLE_CANONICAL_MIN_SCORE or margin < TITLE_CANONICAL_MIN_MARGIN:
                group_records.append({"axis":"title", "group_key":title_key, "candidates":len(cand), "top_score":top_score, "margin":margin, "applied":False, "top_key":cand.content_key.iloc[0]})
                continue
            top_key = cand.content_key.iloc[0]
            pos_idx = g._pos.to_numpy(dtype=int)
            new = (g.content_key.to_numpy() == top_key).astype(np.int8)
            labels[pos_idx] = new
            v4_forced[pos_idx] = new
            v4_rule[pos_idx] = "test_title_canonical"
            locked[pos_idx] = True
            group_records.append({"axis":"title", "group_key":title_key, "candidates":len(cand), "top_score":top_score, "margin":margin, "applied":True, "top_key":top_key})

    # Hard invariant: V4 must never override a V2 graph-forced row.
    immutable = base_forced >= 0
    if not np.array_equal(labels[immutable], base_labels[immutable]):
        raise RuntimeError("V4 attempted to override frozen V2 graph decisions")
    return labels, v4_rule, v4_forced, pd.DataFrame(group_records)


def apply_v5_rescue(v4_labels, v4_rule, v3_blend, v2_global_e):
    """Apply only the pre-agreed asymmetric V5 rescue. Structural V4 decisions are immutable."""
    labels = np.asarray(v4_labels, dtype=np.int8).copy()
    rules = np.asarray(v4_rule, dtype=object)
    v3_blend = np.asarray(v3_blend, dtype=float)
    v2_global_e = np.asarray(v2_global_e, dtype=float)
    if not (len(labels) == len(rules) == len(v3_blend) == len(v2_global_e)):
        raise ValueError("V5 input length mismatch")
    if not (np.isfinite(v3_blend).all() and np.isfinite(v2_global_e).all()):
        raise ValueError("V5 received non-finite score")

    eligible = (
        (labels == 0)
        & (rules == "v2_fallback")
        & (v3_blend >= V5_BLEND_MIN)
        & (v2_global_e >= V5_GLOBAL_ENTAIL_MIN)
    )
    before = labels.copy()
    labels[eligible] = 1

    # Hard invariants: V5 is positive-rescue only and cannot touch structural rows.
    if np.any((before == 1) & (labels == 0)):
        raise RuntimeError("V5 attempted a forbidden 1->0 override")
    if np.any(eligible & (rules != "v2_fallback")):
        raise RuntimeError("V5 attempted to override a structural V4 decision")
    return labels, eligible


Frozen V2 meta threshold recovered from OOF: 0.596982825 | macro_f1=0.836466


## Validation A — component-disjoint transductive rescue audit

In [7]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n)); self.r = [0] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]: self.r[ra] += 1


def build_component_groups(df):
    uf = UnionFind(len(df))
    for col in ["title_key", "content_key"]:
        first = {}
        for i, key in enumerate(df[col].tolist()):
            if key in first: uf.union(i, first[key])
            else: first[key] = i
    roots = [uf.find(i) for i in range(len(df))]
    mapping = {r:j for j,r in enumerate(dict.fromkeys(roots))}
    return np.asarray([mapping[r] for r in roots], dtype=np.int32)


def base_v2_for_query(query_df, ref_df, meta_prob):
    semantic = (np.asarray(meta_prob, dtype=float) >= V2_META_THRESHOLD).astype(np.int8)
    rules, forced = frozen_v2_graph_decisions(query_df, ref_df, meta_prob)
    labels = semantic.copy()
    m = forced >= 0
    labels[m] = forced[m]
    return labels, rules, forced



groups = build_component_groups(train)
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_rows = []
rescue_records = []
for fold, (tr_idx, va_idx) in enumerate(cv.split(train, train.label.values, groups=groups)):
    assert set(groups[tr_idx]).isdisjoint(set(groups[va_idx]))
    ref = train.iloc[tr_idx].reset_index(drop=True)
    qry = train.iloc[va_idx].reset_index(drop=True)
    yq = train.label.to_numpy(dtype=np.int8)[va_idx]

    base, _, forced = base_v2_for_query(qry, ref, v2_oof_meta[va_idx])
    v4, v4_rules, _, _ = apply_transductive_v4(qry, base, forced, rank_oof[va_idx], ref)
    v5, rescue_mask = apply_v5_rescue(v4, v4_rules, v3_oof_blend[va_idx], v2_oof_global[va_idx])

    base_f1 = f1_score(yq, base, average="macro", zero_division=0)
    v4_f1 = f1_score(yq, v4, average="macro", zero_division=0)
    v5_f1 = f1_score(yq, v5, average="macro", zero_division=0)
    n_rescue = int(rescue_mask.sum())
    rescue_precision = float(yq[rescue_mask].mean()) if n_rescue else np.nan
    f0, f1 = f1_score(yq, v5, labels=[0,1], average=None, zero_division=0)

    fold_rows.append({
        "fold": fold,
        "rows": len(va_idx),
        "base_v2_macro_f1": base_f1,
        "v4_macro_f1": v4_f1,
        "v5_macro_f1": v5_f1,
        "gain_v5_vs_v4": v5_f1 - v4_f1,
        "v5_f1_0": f0,
        "v5_f1_1": f1,
        "rescues": n_rescue,
        "rescue_precision": rescue_precision,
    })
    if n_rescue:
        for local_i in np.where(rescue_mask)[0]:
            global_i = va_idx[local_i]
            rescue_records.append({
                "fold": fold,
                "id": train.id.iloc[global_i],
                "label": int(yq[local_i]),
                "v3_blend_prob": float(v3_oof_blend[global_i]),
                "v2_global_e": float(v2_oof_global[global_i]),
                "v2_local_max_c": float(v2_oof.local_max_c.iloc[global_i]),
            })

component_metrics = pd.DataFrame(fold_rows)
rescue_validation = pd.DataFrame(rescue_records)
print("Component-disjoint V5 rescue validation")
display(component_metrics)
if len(rescue_validation):
    display(rescue_validation)

valid_prec = component_metrics.rescue_precision.dropna()
print("Mean V4:", component_metrics.v4_macro_f1.mean())
print("Mean V5:", component_metrics.v5_macro_f1.mean())
print("Mean gain:", component_metrics.gain_v5_vs_v4.mean())
print("Total rescues:", int(component_metrics.rescues.sum()))
print("Observed rescue precision:", float(rescue_validation.label.mean()) if len(rescue_validation) else np.nan)

# Agreed V5 acceptance gates.
assert (component_metrics.gain_v5_vs_v4 >= -1e-12).all(), "V5 regressed in a component-disjoint fold"
assert (valid_prec >= 0.95).all(), "V5 rescue precision below the agreed 95% gate"
assert int(component_metrics.rescues.sum()) > 0, "V5 rescue validation selected no rows"


Component-disjoint V5 rescue validation


,fold,rows,base_v2_macro_f1,v4_macro_f1,v5_macro_f1,gain_v5_vs_v4,v5_f1_0,v5_f1_1,rescues,rescue_precision
0,0,4798,0.847747,0.885052,0.893135,0.008083,0.809011,0.977259,18,1.0
1,1,4807,0.814714,0.890470,0.890470,0.000000,0.802905,0.978035,0,NaN
2,2,4792,0.843990,0.896858,0.899939,0.003081,0.822326,0.977553,7,1.0


,fold,id,label,v3_blend_prob,v2_global_e,v2_local_max_c
0,0,tr00718,1,0.743794,0.556895,0.942671
1,0,tr00807,1,0.702594,0.998401,0.997218
2,0,tr01212,1,0.711983,0.998287,0.953353
3,0,tr01546,1,0.752229,0.601071,0.977756
4,0,tr01553,1,0.707277,0.997815,0.983759
5,0,tr01792,1,0.759747,0.687333,0.874660
6,0,tr02949,1,0.698333,0.975903,0.977271
7,0,tr05019,1,0.673225,0.705803,0.995832
8,0,tr05271,1,0.759853,0.515315,0.979866
9,0,tr06829,1,0.697269,0.996148,0.932286


Mean V4: 0.8907932400047347
Mean V5: 0.8945147092914109
Mean gain: 0.00372146928667616
Total rescues: 25
Observed rescue precision: 1.0


## Validation B — 100 repeated pseudo-competition splits

In [8]:

y_all = train.label.to_numpy(dtype=np.int8)
idx_all = np.arange(len(train))
pseudo_rows = []
for k in range(N_PSEUDO_SPLITS):
    tr_idx, q_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED + k, stratify=y_all)
    ref = train.iloc[tr_idx].reset_index(drop=True)
    qry = train.iloc[q_idx].reset_index(drop=True)

    base, _, forced = base_v2_for_query(qry, ref, v2_oof_meta[q_idx])
    v4, v4_rules, _, _ = apply_transductive_v4(qry, base, forced, rank_oof[q_idx], ref)
    v5, rescue_mask = apply_v5_rescue(v4, v4_rules, v3_oof_blend[q_idx], v2_oof_global[q_idx])
    yq = y_all[q_idx]

    v4_f1 = f1_score(yq, v4, average="macro", zero_division=0)
    v5_f1 = f1_score(yq, v5, average="macro", zero_division=0)
    n_rescue = int(rescue_mask.sum())
    rescue_precision = float(yq[rescue_mask].mean()) if n_rescue else np.nan
    pseudo_rows.append({
        "split": k,
        "v4_macro_f1": v4_f1,
        "v5_macro_f1": v5_f1,
        "gain": v5_f1 - v4_f1,
        "rescues": n_rescue,
        "rescue_precision": rescue_precision,
    })

pseudo_metrics = pd.DataFrame(pseudo_rows)
wins = int((pseudo_metrics.gain > 1e-12).sum())
ties = int((pseudo_metrics.gain.abs() <= 1e-12).sum())
losses = int((pseudo_metrics.gain < -1e-12).sum())
print(f"Pseudo-competition V5: wins={wins} ties={ties} losses={losses} / {N_PSEUDO_SPLITS}")
print("Mean V4 :", pseudo_metrics.v4_macro_f1.mean())
print("Mean V5 :", pseudo_metrics.v5_macro_f1.mean())
print("Mean gain:", pseudo_metrics.gain.mean())
print("Mean rescues:", pseudo_metrics.rescues.mean())
display(pseudo_metrics.describe())

# Agreed V5 acceptance gate: high repeated-split stability.
assert wins >= 90, f"V5 pseudo-competition stability failed: only {wins}/{N_PSEUDO_SPLITS} wins"
assert losses == 0, f"V5 pseudo-competition produced {losses} losing splits"
assert pseudo_metrics.gain.mean() > 0, "V5 pseudo-competition mean gain is not positive"


Pseudo-competition V5: wins=99 ties=1 losses=0 / 100
Mean V4 : 0.918311008625545
Mean V5 : 0.9218042162280773
Mean gain: 0.003493207602532259
Mean rescues: 4.31


,split,v4_macro_f1,v5_macro_f1,gain,rescues,rescue_precision
count,100.000000,100.000000,100.000000,100.000000,100.000000,99.0
mean,49.500000,0.918311,0.921804,0.003493,4.310000,1.0
std,29.011492,0.007501,0.007675,0.001412,1.739006,0.0
min,0.000000,0.902187,0.904430,0.000000,0.000000,1.0
25%,24.750000,0.912949,0.916577,0.002442,3.000000,1.0
50%,49.500000,0.917413,0.921163,0.003290,4.000000,1.0
75%,74.250000,0.922904,0.925854,0.004717,6.000000,1.0
max,99.000000,0.937766,0.940067,0.008050,10.000000,1.0


## V5 test inference — frozen V4 champion + rescue only

In [9]:

# Reconstruct the exact V4 champion from frozen V2/V3 artifacts.
base_v2_label = v2_test.label.to_numpy(dtype=np.int8)
base_v2_forced = v2_test.forced_label.to_numpy(dtype=np.int8)
final_v4_label, v4_rule, v4_forced, canonical_groups = apply_transductive_v4(
    test, base_v2_label, base_v2_forced, rank_test, train
)

changes_v4_vs_v2 = int((final_v4_label != base_v2_label).sum())
assert changes_v4_vs_v2 == EXPECTED_V4_CHANGES_VS_V2, (
    f"Frozen V4 parity drift: expected {EXPECTED_V4_CHANGES_VS_V2} changes vs V2, got {changes_v4_vs_v2}"
)
assert np.array_equal(final_v4_label[base_v2_forced >= 0], base_v2_label[base_v2_forced >= 0])

# Apply only the pre-agreed asymmetric rescue.
final_v5_label, rescue_mask = apply_v5_rescue(final_v4_label, v4_rule, v3_test_blend, v2_test_global)
rescue_count = int(rescue_mask.sum())
changes_v5_vs_v4 = int((final_v5_label != final_v4_label).sum())

print("V2 distribution:", pd.Series(base_v2_label).value_counts().sort_index().to_dict())
print("V4 distribution:", pd.Series(final_v4_label).value_counts().sort_index().to_dict())
print("V5 distribution:", pd.Series(final_v5_label).value_counts().sort_index().to_dict())
print("V4 changes vs V2:", changes_v4_vs_v2)
print("V5 rescues vs V4:", rescue_count)

# Hard methodology-drift guards.
assert rescue_count == EXPECTED_V5_RESCUES, f"Expected {EXPECTED_V5_RESCUES} V5 rescues, got {rescue_count}; aborting"
assert changes_v5_vs_v4 == EXPECTED_V5_RESCUES
assert np.all(final_v4_label[rescue_mask] == 0) and np.all(final_v5_label[rescue_mask] == 1)
assert np.all(np.asarray(v4_rule, dtype=object)[rescue_mask] == "v2_fallback")
assert not np.any(rescue_mask & (base_v2_forced >= 0))
assert not np.any(rescue_mask & (v4_forced >= 0))

changed_v5 = pd.DataFrame({
    "id": test.id,
    "title": test.title,
    "v1_meta_prob": v3_test.v1_meta_prob.to_numpy(dtype=float),
    "v2_meta_prob": v2_test_meta,
    "v3_blend_prob": v3_test_blend,
    "v2_global_e": v2_test_global,
    "v2_local_max_c": v2_test.local_max_c.to_numpy(dtype=float),
    "v2_graph_rule": v2_test.graph_rule.astype(str),
    "v2_forced_label": base_v2_forced,
    "v4_rule": v4_rule,
    "v4_forced_label": v4_forced,
    "v4_label": final_v4_label,
    "v5_label": final_v5_label,
    "rescue_reason": np.where(rescue_mask, "v3>=0.67_and_v2_global_e>=0.50", ""),
}).loc[rescue_mask].reset_index(drop=True)

display(changed_v5)


V2 distribution: {0: 369, 1: 3234}
V4 distribution: {0: 383, 1: 3220}
V5 distribution: {0: 379, 1: 3224}
V4 changes vs V2: 16
V5 rescues vs V4: 4


,id,title,v1_meta_prob,v2_meta_prob,v3_blend_prob,v2_global_e,v2_local_max_c,v2_graph_rule,v2_forced_label,v4_rule,v4_forced_label,v4_label,v5_label,rescue_reason
0,te00907,Dinkes Lansia di Jakarta Mungkin Divaksin Pfizer Vaksinnya Tiba Maret 2021 kumparan com,0.815528,0.568863,0.679862,0.700567,0.944025,semantic,-1,v2_fallback,-1,0,1,v3>=0.67_and_v2_global_e>=0.50
1,te01474,ITAGI Jelaskan Kenapa Ada Efek Lokal usai Disuntik Vaksin Sinovac kumparan com,0.882415,0.561334,0.705821,0.992303,0.931978,semantic,-1,v2_fallback,-1,0,1,v3>=0.67_and_v2_global_e>=0.50
2,te01967,ITAGI Jelaskan Kenapa Ada Efek Lokal usai Disuntik Vaksin Sinovac kumparan com,0.882415,0.561334,0.705821,0.992303,0.931978,semantic,-1,v2_fallback,-1,0,1,v3>=0.67_and_v2_global_e>=0.50
3,te03144,Kata RSSA Malang soal Foto Pasien COVID 19 Tertahan di IGD yang Beredar,0.888168,0.594358,0.726573,0.993542,0.928916,semantic,-1,v2_fallback,-1,0,1,v3>=0.67_and_v2_global_e>=0.50


## Export submission and complete audit bundle

In [10]:

submission = sample[["id"]].copy()
submission["label"] = final_v5_label.astype(int)
assert submission.id.astype(str).tolist() == test.id.astype(str).tolist()
assert set(submission.label.unique()).issubset({0,1})
assert len(submission) == EXPECTED_TEST_ROWS

submission_path = OUTPUT_DIR / "submission_v5.csv"
test_scores_path = OUTPUT_DIR / "test_scores_v5.csv"
changes_path = OUTPUT_DIR / "changed_rows_v5.csv"
component_path = OUTPUT_DIR / "component_rescue_metrics_v5.csv"
rescue_validation_path = OUTPUT_DIR / "rescue_validation_v5.csv"
pseudo_path = OUTPUT_DIR / "pseudo_competition_v5.csv"
profile_path = OUTPUT_DIR / "structural_profile_v5.csv"
groups_path = OUTPUT_DIR / "canonical_groups_v5.csv"
config_path = OUTPUT_DIR / "config_v5.json"
zip_path = OUTPUT_DIR / "results_v5.zip"

submission.to_csv(submission_path, index=False)
pd.DataFrame({
    "id": test.id,
    "v1_meta_prob": v3_test.v1_meta_prob.to_numpy(dtype=float),
    "v2_meta_prob": v2_test_meta,
    "v3_blend_prob": v3_test_blend,
    "v2_global_e": v2_test_global,
    "v2_local_max_c": v2_test.local_max_c.to_numpy(dtype=float),
    "v2_graph_rule": v2_test.graph_rule.astype(str),
    "v2_forced_label": base_v2_forced.astype(int),
    "v2_label": base_v2_label.astype(int),
    "v4_rule": v4_rule,
    "v4_forced_label": v4_forced.astype(int),
    "v4_label": final_v4_label.astype(int),
    "v5_rescue": rescue_mask.astype(int),
    "label": final_v5_label.astype(int),
}).to_csv(test_scores_path, index=False)
changed_v5.to_csv(changes_path, index=False)
component_metrics.to_csv(component_path, index=False)
rescue_validation.to_csv(rescue_validation_path, index=False)
pseudo_metrics.to_csv(pseudo_path, index=False)
profile.to_csv(profile_path, index=False)
canonical_groups.to_csv(groups_path, index=False)

config = {
    "version": "v5-asymmetric-high-confidence-global-rescue",
    "training": False,
    "seed": SEED,
    "base_system": "frozen-v4-champion",
    "public_lb_reference": {"v1":PUBLIC_LB_V1,"v2":PUBLIC_LB_V2,"v3":PUBLIC_LB_V3,"v4":PUBLIC_LB_V4},
    "rescue": {
        "direction": EXPECTED_V5_DIRECTION,
        "required_v4_rule": "v2_fallback",
        "v3_blend_min": V5_BLEND_MIN,
        "v2_global_entail_min": V5_GLOBAL_ENTAIL_MIN,
    },
    "v4_changes_vs_v2": changes_v4_vs_v2,
    "v5_rescues_vs_v4": rescue_count,
    "component_mean_v4": float(component_metrics.v4_macro_f1.mean()),
    "component_mean_v5": float(component_metrics.v5_macro_f1.mean()),
    "component_mean_gain": float(component_metrics.gain_v5_vs_v4.mean()),
    "observed_component_rescue_precision": float(rescue_validation.label.mean()) if len(rescue_validation) else None,
    "pseudo_splits": N_PSEUDO_SPLITS,
    "pseudo_wins": int((pseudo_metrics.gain > 1e-12).sum()),
    "pseudo_ties": int((pseudo_metrics.gain.abs() <= 1e-12).sum()),
    "pseudo_losses": int((pseudo_metrics.gain < -1e-12).sum()),
    "pseudo_mean_gain": float(pseudo_metrics.gain.mean()),
    "test_negative_count": int((final_v5_label == 0).sum()),
}
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

files = [submission_path, test_scores_path, changes_path, component_path, rescue_validation_path,
         pseudo_path, profile_path, groups_path, config_path]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        zf.write(p, arcname=p.name)

print("Saved:")
for p in files + [zip_path]:
    print(" -", p)
print("\nSubmission preview:")
display(submission.head())
print("Final V5 negative count:", int((final_v5_label == 0).sum()))
print("V5 rescue count:", rescue_count)


Saved:
 - /kaggle/working/submission_v5.csv
 - /kaggle/working/test_scores_v5.csv
 - /kaggle/working/changed_rows_v5.csv
 - /kaggle/working/component_rescue_metrics_v5.csv
 - /kaggle/working/rescue_validation_v5.csv
 - /kaggle/working/pseudo_competition_v5.csv
 - /kaggle/working/structural_profile_v5.csv
 - /kaggle/working/canonical_groups_v5.csv
 - /kaggle/working/config_v5.json
 - /kaggle/working/results_v5.zip

Submission preview:


,id,label
0,te00000,0
1,te00001,1
2,te00002,1
3,te00003,1
4,te00004,1


Final V5 negative count: 379
V5 rescue count: 4
